In [3]:
"""
AI Language Translation Tool
============================
A desktop GUI app that translates text using the free MyMemory API.
Features: 15 languages, swap, copy, text-to-speech, history.
"""

import tkinter as tk
from tkinter import ttk, messagebox
import urllib.request
import urllib.parse
import json
import pyperclip
import pyttsx3

# ========== CONFIGURATION ==========
LANGUAGES = {
    "English": "en",
    "Spanish": "es",
    "French": "fr",
    "German": "de",
    "Italian": "it",
    "Portuguese": "pt",
    "Russian": "ru",
    "Japanese": "ja",
    "Korean": "ko",
    "Chinese": "zh",
    "Hindi": "hi",
    "Arabic": "ar",
    "Turkish": "tr",
    "Dutch": "nl",
    "Polish": "pl"
}

API_URL = "https://api.mymemory.translated.net/get?q={text}&langpair={src}|{tgt}"


class TranslationApp:
    def __init__(self, root):
        self.root = root
        self.root.title("🌐 AI Translation Tool")
        self.root.geometry("700x550")
        self.root.configure(bg="#1a1a2e")
        self.root.resizable(False, False)

        # TTS Engine
        self.tts_engine = pyttsx3.init()
        self.tts_engine.setProperty("rate", 160)

        self.history = []

        self._build_ui()

    def _build_ui(self):
        # Header
        header = tk.Frame(self.root, bg="#1a1a2e")
        header.pack(pady=15)

        tk.Label(header, text="AI Translation Tool", font=("Segoe UI", 22, "bold"),
                 fg="#00d4aa", bg="#1a1a2e").pack()
        tk.Label(header, text="Powered by MyMemory API — No API key required",
                 font=("Segoe UI", 10), fg="#8892b0", bg="#1a1a2e").pack()

        # Language Selection Row
        lang_frame = tk.Frame(self.root, bg="#1a1a2e")
        lang_frame.pack(pady=10)

        self.src_lang = ttk.Combobox(lang_frame, values=list(LANGUAGES.keys()),
                                     width=18, state="readonly", font=("Segoe UI", 11))
        self.src_lang.set("English")
        self.src_lang.grid(row=0, column=0, padx=10)

        swap_btn = tk.Button(lang_frame, text="⇄", font=("Segoe UI", 14, "bold"),
                             bg="#0f3460", fg="#00d4aa", width=3,
                             command=self._swap_langs, cursor="hand2",
                             activebackground="#00d4aa", activeforeground="#1a1a2e")
        swap_btn.grid(row=0, column=1, padx=10)

        self.tgt_lang = ttk.Combobox(lang_frame, values=list(LANGUAGES.keys()),
                                     width=18, state="readonly", font=("Segoe UI", 11))
        self.tgt_lang.set("Spanish")
        self.tgt_lang.grid(row=0, column=2, padx=10)

        # Input Text
        input_frame = tk.Frame(self.root, bg="#1a1a2e")
        input_frame.pack(pady=5)

        tk.Label(input_frame, text="Input Text", font=("Segoe UI", 11, "bold"),
                 fg="#ccd6f6", bg="#1a1a2e").pack(anchor="w", padx=10)

        self.input_box = tk.Text(input_frame, height=6, width=70, font=("Segoe UI", 12),
                                 bg="#0f3460", fg="#e0e0e0", insertbackground="#00d4aa",
                                 relief="flat", padx=10, pady=10, wrap="word")
        self.input_box.pack(padx=10, pady=5)

        self.char_label = tk.Label(input_frame, text="0 / 500 characters",
                                   font=("Segoe UI", 9), fg="#5a6a8a", bg="#1a1a2e")
        self.char_label.pack(anchor="e", padx=15)

        self.input_box.bind("<KeyRelease>", self._update_char_count)

        # Translate Button
        trans_btn = tk.Button(self.root, text="TRANSLATE", font=("Segoe UI", 14, "bold"),
                              bg="#00d4aa", fg="#1a1a2e", width=20, cursor="hand2",
                              command=self._translate, activebackground="#00a8e8")
        trans_btn.pack(pady=10)

        # Output Text
        output_frame = tk.Frame(self.root, bg="#1a1a2e")
        output_frame.pack(pady=5)

        tk.Label(output_frame, text="Translated Text", font=("Segoe UI", 11, "bold"),
                 fg="#ccd6f6", bg="#1a1a2e").pack(anchor="w", padx=10)

        self.output_box = tk.Text(output_frame, height=6, width=70, font=("Segoe UI", 12),
                                  bg="#0a192f", fg="#e0e0e0", insertbackground="#00d4aa",
                                  relief="flat", padx=10, pady=10, wrap="word", state="disabled")
        self.output_box.pack(padx=10, pady=5)

        # Action Buttons
        btn_frame = tk.Frame(self.root, bg="#1a1a2e")
        btn_frame.pack(pady=10)

        tk.Button(btn_frame, text="📋 Copy", font=("Segoe UI", 11), bg="#0f3460",
                  fg="#ccd6f6", width=12, command=self._copy_output,
                  cursor="hand2").grid(row=0, column=0, padx=8)
        tk.Button(btn_frame, text="🔊 Speak", font=("Segoe UI", 11), bg="#0f3460",
                  fg="#ccd6f6", width=12, command=self._speak_output,
                  cursor="hand2").grid(row=0, column=1, padx=8)
        tk.Button(btn_frame, text="🗑️ Clear", font=("Segoe UI", 11), bg="#0f3460",
                  fg="#ccd6f6", width=12, command=self._clear_all,
                  cursor="hand2").grid(row=0, column=2, padx=8)
        tk.Button(btn_frame, text="📜 History", font=("Segoe UI", 11), bg="#0f3460",
                  fg="#ccd6f6", width=12, command=self._show_history,
                  cursor="hand2").grid(row=0, column=3, padx=8)

        # Status Bar
        self.status = tk.Label(self.root, text="Ready", font=("Segoe UI", 10),
                               fg="#00d4aa", bg="#1a1a2e")
        self.status.pack(pady=5)

    def _update_char_count(self, event=None):
        count = len(self.input_box.get("1.0", "end-1c"))
        if count > 500:
            self.input_box.delete("1.0 + 500 chars", "end")
            count = 500
        self.char_label.config(text=f"{count} / 500 characters")

    def _swap_langs(self):
        src, tgt = self.src_lang.get(), self.tgt_lang.get()
        self.src_lang.set(tgt)
        self.tgt_lang.set(src)

    def _translate(self):
        text = self.input_box.get("1.0", "end-1c").strip()
        if not text:
            messagebox.showwarning("Empty Input", "Please enter text to translate.")
            return

        src_code = LANGUAGES.get(self.src_lang.get(), "en")
        tgt_code = LANGUAGES.get(self.tgt_lang.get(), "es")

        self.status.config(text="Translating...", fg="#00d4aa")
        self.root.update()

        try:
            url = API_URL.format(
                text=urllib.parse.quote(text),
                src=src_code,
                tgt=tgt_code
            )
            with urllib.request.urlopen(url, timeout=10) as response:
                data = json.loads(response.read().decode())

            if data.get("responseStatus") == 200:
                translated = data["responseData"]["translatedText"]
                self.output_box.config(state="normal")
                self.output_box.delete("1.0", "end")
                self.output_box.insert("1.0", translated)
                self.output_box.config(state="disabled")
                self.status.config(text="Translation complete ✓", fg="#00d4aa")

                # Save to history
                self.history.append({
                    "src": self.src_lang.get(),
                    "tgt": self.tgt_lang.get(),
                    "original": text,
                    "translated": translated
                })
            else:
                self.status.config(text=f"Error: {data.get('responseDetails', 'Unknown')}", fg="#ff6b6b")

        except Exception as e:
            self.status.config(text=f"Network error: {str(e)}", fg="#ff6b6b")

    def _copy_output(self):
        text = self.output_box.get("1.0", "end-1c").strip()
        if text:
            pyperclip.copy(text)
            self.status.config(text="Copied to clipboard!", fg="#00d4aa")
            self.root.after(1500, lambda: self.status.config(text="Ready", fg="#00d4aa"))

    def _speak_output(self):
        text = self.output_box.get("1.0", "end-1c").strip()
        if text:
            self.tts_engine.say(text)
            self.tts_engine.runAndWait()

    def _clear_all(self):
        self.input_box.delete("1.0", "end")
        self.output_box.config(state="normal")
        self.output_box.delete("1.0", "end")
        self.output_box.config(state="disabled")
        self.char_label.config(text="0 / 500 characters")
        self.status.config(text="Ready", fg="#00d4aa")

    def _show_history(self):
        hist_win = tk.Toplevel(self.root)
        hist_win.title("Translation History")
        hist_win.geometry("600x400")
        hist_win.configure(bg="#1a1a2e")

        text = tk.Text(hist_win, wrap="word", font=("Segoe UI", 11),
                       bg="#0f3460", fg="#e0e0e0", padx=10, pady=10)
        text.pack(expand=True, fill="both", padx=10, pady=10)

        if not self.history:
            text.insert("1.0", "No translations yet.")
        else:
            for i, h in enumerate(self.history, 1):
                text.insert("end", f"--- Entry {i} ---\n")
                text.insert("end", f"{h['src']} → {h['tgt']}\n")
                text.insert("end", f"Original: {h['original']}\n")
                text.insert("end", f"Translated: {h['translated']}\n\n")

        text.config(state="disabled")


if __name__ == "__main__":
    root = tk.Tk()
    app = TranslationApp(root)
    root.mainloop()